# Fine-tune the groundedness scorer

Trains a 3-way DeBERTa-v3 scorer (supported / contradicted / neutral), fits a
temperature on validation, and writes the artifacts the harness and demo load.

This notebook is a **driver**. The logic lives in `groundcheck.train`, so it is
unit-tested and identical whether you run it locally or on a hosted GPU. Cells
here set up the environment, call `train()`, and read the results.

**What to watch, in order of importance:**

1. **Does the temperature saturate?** The zero-shot baseline's fit ran to the
   search bound, meaning its confidence carried no usable signal. If this model
   does the same, it cannot act as a gate at any threshold, whatever its accuracy
   looks like. That is the result that decides whether v1 works.
2. **F1 on the not-supported class.** The rare, costly class. Plain accuracy can
   look fine while the model never flags anything.
3. **ECE**, before and after calibration.

Baseline to beat (zero-shot, AggreFact): **BAcc 0.606, ECE 0.444** (0.067 after
calibration, but only by discarding the confidence entirely).

## 1. Environment

The install cell is a no-op locally and does the real work on a fresh hosted
runtime, whose disk does not persist.

In [ ]:
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/swgoodman/groundcheck.git"

if importlib.util.find_spec("groundcheck") is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", "groundcheck[scorers]"],
        check=True,
    )

Imports come after the install so a fresh runtime has the package.

In [ ]:
from groundcheck.device import describe_runtime, resolve_compute_device

print(describe_runtime())
print("compute device:", resolve_compute_device())

Gated datasets (LLM-AggreFact) need a read token. Skip if already logged in.

In [ ]:
from huggingface_hub import get_token

if get_token() is None:
    from huggingface_hub import notebook_login

    notebook_login()
else:
    print("token found")

## 2. Config

The mix, the held-out benchmark, and the hyperparameters, all declared in YAML so
the run is reproducible from the file rather than from these cells.

In [ ]:
from pathlib import Path

from groundcheck.train import TrainConfig

CONFIG = next(
    p
    for p in [Path("../configs/train_v1.yaml"), Path("groundcheck/configs/train_v1.yaml")]
    if p.exists()
)
config = TrainConfig.from_yaml(CONFIG)

config.fp16 = resolve_compute_device() == "cuda"
config

## 3. Inspect the mix before training

Class balance and the coarse fraction decide what the loss can actually supervise.
Roughly half the pool marks a claim unsupported without saying whether it
contradicts the passage or merely is not covered by it; those rows supervise the
collapsed decision only. Worth seeing the number rather than assuming it.

In [ ]:
from collections import Counter

from groundcheck.train import encode_labels, load_sources, prepare_training_data

train_examples, provenance = prepare_training_data(config)
eval_examples = load_sources(config.eval_sources)
labels, coarse = encode_labels(train_examples)

print(f"train {len(train_examples):,}   eval {len(eval_examples):,}")
print(f"coarse 3-way labels: {coarse.mean():.1%}")
print("labels:", Counter(e.label for e in train_examples))
print("sources:", Counter(e.meta["dataset"] for e in train_examples))
print("decontamination:", provenance)

## 4. Train

Cross-entropy on rows with a reliable 3-way label, collapsed binary cross-entropy
on coarse rows, inverse-frequency class weights throughout. Best checkpoint is
selected on F1 of the not-supported class, not accuracy.

In [ ]:
from groundcheck.train import train

summary = train(config)
summary

## 5. Calibration

**Read the temperature first.** If it sits at the edge of the search range, a
`RuntimeWarning` will have fired and the value is a clamp rather than a fit: the
optimizer wanted to keep softening, which means the confidence is uninformative
and the scorer cannot gate.

In [ ]:
calibration = summary["calibration"]
print(f"temperature : {calibration['temperature']:.3f}")
print(f"ECE before  : {calibration['ece_before']:.3f}")
print(f"ECE after   : {calibration['ece_after']:.3f}")
print(f"balanced acc: {calibration['balanced_acc']:.3f}")

if calibration["temperature"] > 50:
    print("\nSATURATED: confidence carries little signal. Investigate before shipping a gate.")

## 6. Evaluate through the harness

Same runner and metrics as every other scorer, so this row is comparable to the
zero-shot baseline and to the comparators. AggreFact is the leaderboard-comparable
surface; RAGTruth test is the in-domain diagnostic.

Latency is deliberately left unmeasured here. A number from this machine is not
publishable; run the benchmark on a named reference instance instead.

In [ ]:
from groundcheck.eval import report
from groundcheck.eval.runner import run
from groundcheck.registry import get_dataset
from groundcheck.scorers.nli_zeroshot import NLIZeroShot

tuned = NLIZeroShot(
    model_name=summary["output_dir"],
    temperature=calibration["temperature"],
)
tuned.name = "groundcheck-deberta-v3-base"

baseline = NLIZeroShot()

results = []
for name, split, limit in [("aggrefact", "test", 2000), ("ragtruth", "test", 1000)]:
    examples = get_dataset(name).load(split, limit=limit)
    for scorer in (tuned, baseline):
        results.append(run(scorer, examples, dataset_name=name, split=split))

print(report.to_markdown(results, title="Fine-tuned vs zero-shot"))

## 7. Error analysis

Prioritize **false-supported**: claims the scorer passed that were not grounded.
Those are the failures that reach a user, whereas a false alarm only costs a
review. Read them before writing anything about which weaknesses dominate.

In [ ]:
examples = get_dataset("aggrefact").load("test", limit=2000)
verdicts = tuned.score_batch(examples)

missed = sorted(
    (
        (v.score, e)
        for e, v in zip(examples, verdicts, strict=True)
        if v.supported and not e.supported
    ),
    key=lambda pair: -pair[0],
)

n_notsup = sum(not e.supported for e in examples)
print(f"missed hallucinations: {len(missed)} of {n_notsup} not-supported\n")
for score, example in missed[:10]:
    print(f"[p={score:.3f}] {example.meta['source_dataset']}")
    print(f"  claim  : {example.claim[:160]}")
    print(f"  context: {example.context[:160]}...\n")

## 8. Per-corpus breakdown

AggreFact is 56% RAGTruth, so a single aggregate number is mostly a RAGTruth
number. The breakdown is what shows whether the scorer generalizes or has learned
one corpus.

In [ ]:
from groundcheck.data.aggrefact import AggreFact

corpora = sorted({e.meta["source_dataset"] for e in examples})
per_corpus = []
for corpus in corpora:
    slice_ = AggreFact(source_datasets=(corpus,)).load("test", limit=400)
    if len(slice_) < 50:
        continue
    per_corpus.append(run(tuned, slice_, dataset_name=corpus, split="test"))

print(report.to_markdown(per_corpus, title="Per-corpus"))

## 9. Publish

Push weights, tokenizer, and the fitted temperature. A scorer that reports
calibrated probabilities in a notebook and raw softmax in production is worse than
one that never claimed calibration, so the temperature ships with the model.

Set `hub_repo_id` in the config first.

In [ ]:
# train(config, push_to_hub=True)

## 10. Reproducibility footer

In [ ]:
import json

import datasets
import torch
import transformers

print(
    json.dumps(
        {
            "torch": torch.__version__,
            "transformers": transformers.__version__,
            "datasets": datasets.__version__,
            "seed": config.seed,
            "runtime": summary["runtime"],
            "provenance": summary["provenance"],
        },
        indent=2,
    )
)